<a href="https://colab.research.google.com/github/MeouchyCharbel/Neural-Machine-Translation/blob/feature%2Fevaluation-improvements/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = gpu_memory / (1024 ** 3)

    print("GPU memory:", round(gpu_memory_gb, 2), "GB")
else:
    print("No GPU detected. Check Colab runtime settings.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:

%pip install -q transformers datasets accelerate evaluate sentencepiece sacremoses sacrebleu scikit-learn pandas


In [ ]:

import csv
import glob
import hashlib
import urllib.request
import zipfile
import pandas as pd

DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip"
ZIP_PATH = "fra-eng.zip"
EXTRACT_DIR = "data_raw"

urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with open(ZIP_PATH, "rb") as file:
    dataset_hash = hashlib.file_digest(file, "sha256").hexdigest()

with zipfile.ZipFile(ZIP_PATH) as zip_file:
    zip_file.extractall(EXTRACT_DIR)

text_file = glob.glob(
    f"{EXTRACT_DIR}/**/fra.txt",
    recursive=True
)[0]

pairs = pd.read_csv(
    text_file,
    sep="\t",
    header=None,
    usecols=[0, 1],
    names=["en", "fr"],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

rows_before = len(pairs)

pairs = pairs.dropna(subset=["en", "fr"]).copy()

pairs["en"] = pairs["en"].str.strip()
pairs["fr"] = pairs["fr"].str.strip()

pairs["en_words"] = pairs["en"].str.split().str.len()

pairs = pairs[
    (pairs["en"] != "") &
    (pairs["fr"] != "")
]

pairs = pairs.drop_duplicates(subset=["en", "fr"])
pairs = pairs[pairs["en_words"] <= 15]
pairs = pairs.reset_index(drop=True)

print("Original sentence pairs:", rows_before)
print("Cleaned sentence pairs:", len(pairs))
print("Unique English sentences:", pairs["en"].nunique())
print("Dataset SHA-256:", dataset_hash)

display(pairs[["en", "fr"]].head(5))


Original sentence pairs: 167130
Cleaned sentence pairs: 166340
Unique English sentences: 116198
Dataset SHA-256: 036e7dc4e83d393db24720ca0c7b7ddcb4b6b6053960438a5c0e1d6d8f2528f5


,en,fr
0,Go.,Va !
1,Hi.,Salut !
2,Run!,Cours !
3,Run!,Courez !
4,Who?,Qui ?


In [ ]:

import re
import unicodedata
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def loose_normalize(text):
    text = normalize_text(text)
    text = re.sub(r"[\W_]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

pairs["en_key"] = pairs["en"].apply(normalize_text)
pairs["fr_key"] = pairs["fr"].apply(normalize_text)
pairs["en_loose"] = pairs["en"].apply(loose_normalize)
pairs["fr_loose"] = pairs["fr"].apply(loose_normalize)

parent = list(range(len(pairs)))
size = [1] * len(pairs)

def find(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a == root_b:
        return

    if size[root_a] < size[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a
    size[root_a] += size[root_b]

seen = {}

for i, (english, french) in enumerate(
    zip(pairs["en_loose"], pairs["fr_loose"])
):
    for key in [("en", english), ("fr", french)]:
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i

pairs["group_id"] = [
    find(i) for i in range(len(pairs))
]

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, remaining_idx = next(
    first_split.split(
        pairs,
        groups=pairs["group_id"]
    )
)

train_pairs = pairs.iloc[train_idx].reset_index(drop=True)
remaining_pairs = pairs.iloc[remaining_idx].reset_index(drop=True)

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)

val_idx, test_idx = next(
    second_split.split(
        remaining_pairs,
        groups=remaining_pairs["group_id"]
    )
)

val_pairs = remaining_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = remaining_pairs.iloc[test_idx].reset_index(drop=True)

assert len(train_pairs) == 133017
assert len(val_pairs) == 16616
assert len(test_pairs) == 16707

splits = {
    "Train": train_pairs,
    "Validation": val_pairs,
    "Test": test_pairs
}

columns_to_check = [
    "en_key",
    "fr_key",
    "en_loose",
    "fr_loose",
    "group_id"
]

comparisons = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

for left_name, right_name in comparisons:
    for column in columns_to_check:
        overlap = (
            set(splits[left_name][column])
            & set(splits[right_name][column])
        )

        assert len(overlap) == 0

assert sum(len(df) for df in splits.values()) == len(pairs)

print("DATASET SPLITTING VERIFIED")
print("Training:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Testing:", len(test_pairs))
print("Connected groups:", pairs["group_id"].nunique())
print("Largest group:", pairs["group_id"].value_counts().max())
print("All exact and normalized overlap checks passed.")


DATASET SPLITTING VERIFIED
Training: 133017
Validation: 16616
Testing: 16707
Connected groups: 108720
Largest group: 96
All exact and normalized overlap checks passed.


In [ ]:

import hashlib
import json

SEED = 42
TRAIN_SAMPLE_SIZE = 20000
VALIDATION_SAMPLE_SIZE = 1000

training_subset = train_pairs.sample(
    n=min(TRAIN_SAMPLE_SIZE, len(train_pairs)),
    random_state=SEED
).reset_index(drop=True)

validation_subset = val_pairs.sample(
    n=min(VALIDATION_SAMPLE_SIZE, len(val_pairs)),
    random_state=SEED
).reset_index(drop=True)

train_groups = set(training_subset["group_id"])
validation_groups = set(validation_subset["group_id"])
test_groups = set(test_pairs["group_id"])

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

def dataset_fingerprint(data):
    records = sorted(
        zip(data["en"].tolist(), data["fr"].tolist())
    )

    content = json.dumps(
        records,
        ensure_ascii=False,
        separators=(",", ":")
    )

    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()

train_fingerprint = dataset_fingerprint(training_subset)
validation_fingerprint = dataset_fingerprint(validation_subset)

print("FINE-TUNING DATASET")
print("Training pairs:", len(training_subset))
print("Validation pairs:", len(validation_subset))
print("Random seed:", SEED)
print("Training fingerprint:", train_fingerprint)
print("Validation fingerprint:", validation_fingerprint)
print("Group overlap checks: Passed")

display(
    training_subset[["en", "fr"]].head(5)
)


FINE-TUNING DATASET
Training pairs: 20000
Validation pairs: 1000
Random seed: 42
Training fingerprint: 99887fd07f9b20eb72d50473b9608cd565f7ab5e615e3ed89384765bfb3a74b3
Validation fingerprint: 1e4e9761bcf231fbe6e0cf769a63d42a34ea8081da7556825ac0193120f77d13
Group overlap checks: Passed


,en,fr
0,"I just think you should be careful, that's all.",Je pense simplement que vous devriez être prud...
1,Who asked you to do that?,Qui t'a demandé de faire ça ?
2,I always thought Tom would outlive me.,J'ai toujours pensé que Tom me survivrait.
3,I like to eat apples.,J'aime manger des pommes.
4,Where would you want to live?,Où voudrais-tu vivre ?


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "Helsinki-NLP/opus-mt-en-fr"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "GPU is required for this experiment."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model = model.to(device)

model.train()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model loaded successfully!")
print("Device:", device)
print("Model:", MODEL_NAME)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  301MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  301MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda
Model: Helsinki-NLP/opus-mt-en-fr
Total parameters: 75,133,952
Trainable parameters: 75,133,952
GPU memory allocated: 0.28 GB


In [ ]:

from datasets import Dataset

MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

train_dataset = Dataset.from_pandas(
    training_subset[["en", "fr"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_subset[["en", "fr"]],
    preserve_index=False
)

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["en"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    targets = tokenizer(
        text_target=examples["fr"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["en", "fr"],
    desc="Tokenizing training data"
)

tokenized_validation = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["en", "fr"],
    desc="Tokenizing validation data"
)

print("TOKENIZATION COMPLETED")
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))

print("\nFirst English sentence:")
print(training_subset.iloc[0]["en"])

print("\nFirst French reference:")
print(training_subset.iloc[0]["fr"])

print("\nDecoded input tokens:")
print(
    tokenizer.decode(
        tokenized_train[0]["input_ids"],
        skip_special_tokens=True
    )
)

print("\nDecoded target tokens:")
print(
    tokenizer.decode(
        tokenized_train[0]["labels"],
        skip_special_tokens=True
    )
)

print("\nFirst input token count:", len(tokenized_train[0]["input_ids"]))
print("First target token count:", len(tokenized_train[0]["labels"]))


Tokenizing training data:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/1000 [00:00<?, ? examples/s]

TOKENIZATION COMPLETED
Training examples: 20000
Validation examples: 1000

First English sentence:
I just think you should be careful, that's all.

First French reference:
Je pense simplement que vous devriez être prudent, c'est tout.

Decoded input tokens:
I just think you should be careful, that's all.

Decoded target tokens:
Je pense simplement que vous devriez être prudent, c'est tout.

First input token count: 14
First target token count: 15


In [ ]:

from pathlib import Path
from google.colab import drive
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

drive.mount("/content/drive")

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/NMT_Finetuning/pilot"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    max_steps=100,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    report_to="none",
    seed=42,
    data_seed=42,
    optim="adamw_torch"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

sample_batch = data_collator(
    [tokenized_train[0], tokenized_train[1]]
)

print("TRAINING CONFIGURATION READY")
print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))
print("Pilot optimizer steps:", training_args.max_steps)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)

print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print("Mixed precision:", training_args.fp16)
print("Learning rate:", training_args.learning_rate)
print("Checkpoint directory:", CHECKPOINT_DIR)

print("\nDYNAMIC PADDING CHECK")

for name, value in sample_batch.items():
    print(name, tuple(value.shape))

print("\nTrainer initialized successfully.")


Mounted at /content/drive
TRAINING CONFIGURATION READY
GPU: Tesla T4
Training examples: 20000
Validation examples: 1000
Pilot optimizer steps: 100
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Mixed precision: True
Learning rate: 2e-05
Checkpoint directory: /content/drive/MyDrive/NMT_Finetuning/pilot

DYNAMIC PADDING CHECK
input_ids (2, 16)
attention_mask (2, 16)
labels (2, 16)
decoder_input_ids (2, 16)

Trainer initialized successfully.


In [ ]:

import time
import torch

torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

train_output = trainer.train()

training_time = time.perf_counter() - start_time

print("\nPILOT FINE-TUNING COMPLETED")

print("Completed steps:", trainer.state.global_step)

print(
    "Average training loss:",
    round(train_output.training_loss, 4)
)

print(
    "Best validation loss:",
    trainer.state.best_metric
)

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Total training time:",
    round(training_time, 2),
    "seconds"
)

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)


Step,Training Loss,Validation Loss
50,2.317148,0.565715
100,2.274933,0.555836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



PILOT FINE-TUNING COMPLETED
Completed steps: 100
Average training loss: 2.4508
Best validation loss: 0.5558357834815979
Best checkpoint: /content/drive/MyDrive/NMT_Finetuning/pilot/checkpoint-100
Total training time: 84.03 seconds
Peak GPU memory: 1.42 GB


In [ ]:

import torch

evaluation_metrics = trainer.evaluate()

current_loss = evaluation_metrics["eval_loss"]
best_loss = trainer.state.best_metric

print("CHECKPOINT VERIFICATION")
print("Best saved validation loss:", round(best_loss, 6))
print("Current validation loss:", round(current_loss, 6))
print("Best checkpoint:", trainer.state.best_model_checkpoint)

if abs(current_loss - best_loss) < 0.05:
    print("Checkpoint validation: PASSED")
else:
    print("Checkpoint validation: NEEDS INVESTIGATION")

model.eval()

sample_english = validation_subset["en"].head(3).tolist()
sample_french = validation_subset["fr"].head(3).tolist()

inputs = tokenizer(
    sample_english,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
).to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        num_beams=1,
        do_sample=False,
        max_new_tokens=64
    )

translations = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

print("\nFINE-TUNED MODEL TRANSLATIONS")

for english, reference, prediction in zip(
    sample_english,
    sample_french,
    translations
):
    print("\nEnglish:", english)
    print("Reference:", reference)
    print("Fine-tuned:", prediction)
    print("-" * 60)


Training Loss,Validation Loss,Step
2.274933,0.555836,100


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CHECKPOINT VERIFICATION
Best saved validation loss: 0.555836
Current validation loss: 0.555836
Best checkpoint: /content/drive/MyDrive/NMT_Finetuning/pilot/checkpoint-100
Checkpoint validation: PASSED

FINE-TUNED MODEL TRANSLATIONS

English: I never make a speech without being nervous.
Reference: Je ne fais jamais un discours sans être nerveux.
Fine-tuned: Je ne fais jamais de discours sans être nerveuse.
------------------------------------------------------------

English: She's fashionable.
Reference: Elle est à la mode.
Fine-tuned: Elle est à la mode.
------------------------------------------------------------

English: Remember to lock the door.
Reference: Rappelle-toi de fermer la porte à clé.
Fine-tuned: N'oubliez pas de verrouiller la porte.
------------------------------------------------------------


In [ ]:

import pandas as pd
import sacrebleu
import torch
import time

RESULTS_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/validation_predictions.csv"
)

comparison_df = pd.read_csv(RESULTS_URL)

expected_validation = val_pairs.drop_duplicates(
    subset="en_key"
).sample(
    n=500,
    random_state=42
).reset_index(drop=True)

assert len(comparison_df) == 500

assert (
    comparison_df["en"].tolist()
    == expected_validation["en"].tolist()
)

assert (
    comparison_df["fr"].tolist()
    == expected_validation["fr"].tolist()
)

trainer.model.eval()

sentences = comparison_df["en"].tolist()
references = comparison_df["fr"].tolist()

batch_size = 8
finetuned_predictions = []

warmup_inputs = tokenizer(
    sentences[:8],
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
).to(device)

with torch.inference_mode():
    trainer.model.generate(
        **warmup_inputs,
        num_beams=1,
        do_sample=False,
        max_new_tokens=64
    )

torch.cuda.synchronize()
start_time = time.perf_counter()

for start in range(0, len(sentences), batch_size):
    batch = sentences[start:start + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    ).to(device)

    with torch.inference_mode():
        outputs = trainer.model.generate(
            **inputs,
            num_beams=1,
            do_sample=False,
            max_new_tokens=64
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    finetuned_predictions.extend(translations)

torch.cuda.synchronize()
finetuned_time = time.perf_counter() - start_time

comparison_df["finetuned_translation"] = finetuned_predictions

original_predictions = comparison_df["transformer_greedy"].tolist()

original_bleu = sacrebleu.corpus_bleu(
    original_predictions,
    [references]
).score

original_chrf = sacrebleu.corpus_chrf(
    original_predictions,
    [references]
).score

finetuned_bleu = sacrebleu.corpus_bleu(
    finetuned_predictions,
    [references]
).score

finetuned_chrf = sacrebleu.corpus_chrf(
    finetuned_predictions,
    [references]
).score

comparison_results_df = pd.DataFrame({
    "Model": [
        "Original Pretrained",
        "Fine-tuned 100 Steps"
    ],
    "BLEU": [
        round(original_bleu, 2),
        round(finetuned_bleu, 2)
    ],
    "chrF": [
        round(original_chrf, 2),
        round(finetuned_chrf, 2)
    ]
})

print("PRETRAINED VS FINE-TUNED")
print("Validation sentences:", len(comparison_df))

display(comparison_results_df)

print("\nQUALITY DIFFERENCES")
print("BLEU difference:", round(finetuned_bleu - original_bleu, 2))
print("chrF difference:", round(finetuned_chrf - original_chrf, 2))

print("\nFINE-TUNED INFERENCE PERFORMANCE")
print("Device:", device)
print("Total time:", round(finetuned_time, 2), "seconds")
print("Sentences/second:", round(len(sentences) / finetuned_time, 2))

changed = comparison_df[
    comparison_df["transformer_greedy"]
    != comparison_df["finetuned_translation"]
]

print("\nTranslations changed:", len(changed), "/", len(comparison_df))

display(
    changed[
        [
            "en",
            "fr",
            "transformer_greedy",
            "finetuned_translation"
        ]
    ].head(10)
)

comparison_df.to_csv(
    "pretrained_vs_finetuned_validation.csv",
    index=False
)


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

PRETRAINED VS FINE-TUNED
Validation sentences: 500


,Model,BLEU,chrF
0,Original Pretrained,55.11,70.40
1,Fine-tuned 100 Steps,55.84,71.62



QUALITY DIFFERENCES
BLEU difference: 0.73
chrF difference: 1.22

FINE-TUNED INFERENCE PERFORMANCE
Device: cuda
Total time: 10.92 seconds
Sentences/second: 45.81

Translations changed: 176 / 500


,en,fr,transformer_greedy,finetuned_translation
0,Look up the words in your dictionary.,Cherchez les mots dans votre dictionnaire.,Regarde les mots de ton dictionnaire.,Regardez les mots de votre dictionnaire.
1,I arrived at the station ten minutes late.,Je suis arrivé à la gare dix minutes en retard.,Je suis arrivé à la gare dix minutes en retard.,Je suis arrivé à la gare dix minutes de retard.
8,I made sure of that.,Je m'en suis assuré.,J'en ai assuré.,J'en ai assuré le bien-être.
10,Why can't we spend more time together?,Pourquoi ne pouvons-nous pas passer plus de te...,Pourquoi ne pas passer plus de temps ensemble ?,Pourquoi ne pouvons-nous pas passer plus de te...
11,Tom seems slightly confused.,Tom semble légèrement confus.,Tom semble un peu confus.,Tom semble légèrement confus.
13,I don't know what's got into her.,Je ne sais pas ce qui lui prend.,Je ne sais pas ce qui lui prend.,Je ne sais pas ce qui lui est arrivé.
29,A long piece of thread easily becomes tangled up.,Un long fil s'emmêle facilement.,Un long morceau de fil se met facilement en ch...,Un long morceau de fil devient facilement ench...
31,I think I've got a touch of fever.,Je pense que j'ai attrapé un peu de fièvre.,Je crois que j'ai une touche de fièvre.,Je pense que j'ai une touche de fièvre.
35,I can't deal with this problem anymore.,Je n'arrive plus à gérer ce problème.,Je ne peux plus gérer ce problème.,Je ne peux plus faire face à ce problème.
36,My mother visits the dentist's every other day...,Ma mère rend visite au dentiste tous les deux ...,Ma mère visite le dentiste tous les deux jours...,Ma mère visite le dentiste tous les deux jours...


In [ ]:

from google.colab import files

comparison_df.to_csv(
    "pretrained_vs_finetuned_validation.csv",
    index=False
)

comparison_results_df.to_csv(
    "pretrained_vs_finetuned_metrics.csv",
    index=False
)

print("COMPARISON RESULTS SAVED")
print("Validation predictions:", len(comparison_df))
print("Models compared:", len(comparison_results_df))

files.download("pretrained_vs_finetuned_validation.csv")
files.download("pretrained_vs_finetuned_metrics.csv")


COMPARISON RESULTS SAVED
Validation predictions: 500
Models compared: 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import numpy as np
import pandas as pd
import sacrebleu
import time

RESULTS_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/pretrained_vs_finetuned_validation.csv"
)

analysis_df = pd.read_csv(RESULTS_URL)

analysis_df = analysis_df.merge(
    val_pairs[["en", "fr", "group_id"]],
    on=["en", "fr"],
    how="left",
    validate="many_to_one"
)

assert len(analysis_df) == 500
assert analysis_df["group_id"].notna().all()

original = analysis_df["transformer_greedy"].tolist()
finetuned = analysis_df["finetuned_translation"].tolist()
references = analysis_df["fr"].tolist()

bleu_metric = sacrebleu.metrics.BLEU()
chrf_metric = sacrebleu.metrics.CHRF()

def calculate_scores(indices):
    ref = [references[i] for i in indices]
    old = [original[i] for i in indices]
    new = [finetuned[i] for i in indices]

    bleu_difference = (
        bleu_metric.corpus_score(new, [ref]).score
        - bleu_metric.corpus_score(old, [ref]).score
    )

    chrf_difference = (
        chrf_metric.corpus_score(new, [ref]).score
        - chrf_metric.corpus_score(old, [ref]).score
    )

    return bleu_difference, chrf_difference

group_values = analysis_df["group_id"].to_numpy()
unique_groups = np.unique(group_values)

group_rows = [
    np.flatnonzero(group_values == group)
    for group in unique_groups
]

rng = np.random.default_rng(42)
N_BOOTSTRAP = 1000

bootstrap_differences = np.zeros((N_BOOTSTRAP, 2))

start_time = time.perf_counter()

for iteration in range(N_BOOTSTRAP):
    sampled_groups = rng.integers(
        0,
        len(group_rows),
        size=len(group_rows)
    )

    sampled_indices = np.concatenate(
        [group_rows[i] for i in sampled_groups]
    )

    bootstrap_differences[iteration] = calculate_scores(
        sampled_indices
    )

observed = calculate_scores(
    np.arange(len(analysis_df))
)

confidence_intervals = np.percentile(
    bootstrap_differences,
    [2.5, 97.5],
    axis=0
)

bootstrap_results_df = pd.DataFrame({
    "Metric": ["BLEU", "chrF"],
    "Observed Improvement": np.round(observed, 3),
    "95% CI Lower": np.round(confidence_intervals[0], 3),
    "95% CI Upper": np.round(confidence_intervals[1], 3)
})

bootstrap_results_df["CI Excludes Zero"] = (
    (bootstrap_results_df["95% CI Lower"] > 0)
    |
    (bootstrap_results_df["95% CI Upper"] < 0)
)

print("PAIRED GROUP BOOTSTRAP ANALYSIS")
print("Sentences:", len(analysis_df))
print("Connected groups:", len(unique_groups))
print("Bootstrap repetitions:", N_BOOTSTRAP)

display(bootstrap_results_df)

print(
    "Analysis time:",
    round(time.perf_counter() - start_time, 2),
    "seconds"
)


PAIRED GROUP BOOTSTRAP ANALYSIS
Sentences: 500
Connected groups: 496
Bootstrap repetitions: 1000


,Metric,Observed Improvement,95% CI Lower,95% CI Upper,CI Excludes Zero
0,BLEU,0.726,-0.654,2.212,False
1,chrF,1.217,0.228,2.315,True


Analysis time: 173.2 seconds


In [ ]:

from google.colab import files

bootstrap_results_df.to_csv(
    "finetuning_bootstrap_results.csv",
    index=False
)

print("STATISTICAL ANALYSIS SAVED")
print("Bootstrap repetitions:", N_BOOTSTRAP)
print("Validation sentences:", len(analysis_df))
print("Connected groups:", len(unique_groups))

display(bootstrap_results_df)

files.download("finetuning_bootstrap_results.csv")


STATISTICAL ANALYSIS SAVED
Bootstrap repetitions: 1000
Validation sentences: 500
Connected groups: 496


,Metric,Observed Improvement,95% CI Lower,95% CI Upper,CI Excludes Zero
0,BLEU,0.726,-0.654,2.212,False
1,chrF,1.217,0.228,2.315,True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import torch
from pathlib import Path
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

PILOT_CHECKPOINT = Path(
    "/content/drive/MyDrive/NMT_Finetuning/pilot/checkpoint-100"
)

EXTENDED_DIR = Path(
    "/content/drive/MyDrive/NMT_Finetuning/extended"
)

assert PILOT_CHECKPOINT.is_dir()
assert (PILOT_CHECKPOINT / "config.json").exists()
assert torch.cuda.is_available()

EXTENDED_DIR.mkdir(parents=True, exist_ok=True)

existing_checkpoints = list(
    EXTENDED_DIR.glob("checkpoint-*")
)

assert not existing_checkpoints, (
    "Extended checkpoints already exist. "
    "Inspect them before starting a new experiment."
)

extended_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(PILOT_CHECKPOINT)
)

extended_model = extended_model.to(device)

extended_args = Seq2SeqTrainingArguments(
    output_dir=str(EXTENDED_DIR),
    num_train_epochs=1.0,
    max_steps=-1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_steps=63,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=125,
    save_strategy="steps",
    save_steps=125,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    report_to="none",
    seed=42,
    data_seed=42,
    optim="adamw_torch"
)

extended_trainer = Seq2SeqTrainer(
    model=extended_model,
    args=extended_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ]
)

print("EXTENDED TRAINING CONFIGURATION READY")
print("Starting checkpoint:", PILOT_CHECKPOINT)
print("Model loaded:", MODEL_NAME)
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))
print("Maximum additional epochs:", extended_args.num_train_epochs)
print("Learning rate:", extended_args.learning_rate)
print("Warmup steps:", extended_args.warmup_steps)
print("Evaluation interval:", extended_args.eval_steps)
print("Checkpoint interval:", extended_args.save_steps)
print("Early stopping patience:", 3)
print("Output directory:", EXTENDED_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Trainer initialized successfully.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

EXTENDED TRAINING CONFIGURATION READY
Starting checkpoint: /content/drive/MyDrive/NMT_Finetuning/pilot/checkpoint-100
Model loaded: Helsinki-NLP/opus-mt-en-fr
Training examples: 20000
Validation examples: 1000
Maximum additional epochs: 1.0
Learning rate: 1e-05
Warmup steps: 63
Evaluation interval: 125
Checkpoint interval: 125
Early stopping patience: 3
Output directory: /content/drive/MyDrive/NMT_Finetuning/extended
GPU: Tesla T4
Trainer initialized successfully.


In [ ]:

import time
import torch

torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

extended_train_output = extended_trainer.train()

extended_training_time = time.perf_counter() - start_time

print("\nEXTENDED FINE-TUNING COMPLETED")

print("Completed optimizer steps:", extended_trainer.state.global_step)

print(
    "Average training loss:",
    round(extended_train_output.training_loss, 4)
)

print(
    "Best validation loss:",
    round(extended_trainer.state.best_metric, 6)
)

print(
    "Best checkpoint:",
    extended_trainer.state.best_model_checkpoint
)

print(
    "Total training time:",
    round(extended_training_time, 2),
    "seconds"
)

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print("\nVALIDATION HISTORY")

for entry in extended_trainer.state.log_history:
    if "eval_loss" in entry:
        print(
            "Step:", entry["step"],
            "| Validation loss:",
            round(entry["eval_loss"], 6)
        )


Step,Training Loss,Validation Loss
125,2.162218,0.545004
250,2.110330,0.533094


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
125,2.162218,0.545004
250,2.110330,0.533094
375,2.375828,0.525507
500,1.920847,0.521328
625,2.235797,0.516788
750,2.123083,0.515457
875,2.165207,0.512915
1000,2.121200,0.511794
1125,1.954214,0.510876
1250,2.068409,0.510601


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



EXTENDED FINE-TUNING COMPLETED
Completed optimizer steps: 1250
Average training loss: 2.1373
Best validation loss: 0.510601
Best checkpoint: /content/drive/MyDrive/NMT_Finetuning/extended/checkpoint-1250
Total training time: 442.0 seconds
Peak GPU memory: 2.25 GB

VALIDATION HISTORY
Step: 125 | Validation loss: 0.545004
Step: 250 | Validation loss: 0.533094
Step: 375 | Validation loss: 0.525507
Step: 500 | Validation loss: 0.521328
Step: 625 | Validation loss: 0.516788
Step: 750 | Validation loss: 0.515457
Step: 875 | Validation loss: 0.512915
Step: 1000 | Validation loss: 0.511794
Step: 1125 | Validation loss: 0.510876
Step: 1250 | Validation loss: 0.510601


In [ ]:

import torch
import math

verification = extended_trainer.evaluate()

saved_best_loss = extended_trainer.state.best_metric
current_loss = verification["eval_loss"]

difference = abs(saved_best_loss - current_loss)

print("EXTENDED CHECKPOINT VERIFICATION")

print("Completed steps:", extended_trainer.state.global_step)

print("Best saved validation loss:", round(saved_best_loss, 6))
print("Recalculated validation loss:", round(current_loss, 6))
print("Absolute difference:", round(difference, 8))

print(
    "Best checkpoint:",
    extended_trainer.state.best_model_checkpoint
)

if math.isfinite(current_loss) and difference < 0.001:
    print("Checkpoint validation: PASSED")
else:
    print("Checkpoint validation: NEEDS INVESTIGATION")

extended_trainer.model.eval()

sample_sentences = validation_subset["en"].head(3).tolist()
sample_references = validation_subset["fr"].head(3).tolist()

inputs = tokenizer(
    sample_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
).to(device)

with torch.inference_mode():
    outputs = extended_trainer.model.generate(
        **inputs,
        num_beams=1,
        do_sample=False,
        max_new_tokens=64
    )

predictions = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

print("\nEXTENDED MODEL TRANSLATIONS")

for english, reference, prediction in zip(
    sample_sentences,
    sample_references,
    predictions
):
    print("\nEnglish:", english)
    print("Reference:", reference)
    print("Extended model:", prediction)
    print("-" * 60)


Training Loss,Validation Loss,Step
2.068409,0.510601,1250


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXTENDED CHECKPOINT VERIFICATION
Completed steps: 1250
Best saved validation loss: 0.510601
Recalculated validation loss: 0.510601
Absolute difference: 0.0
Best checkpoint: /content/drive/MyDrive/NMT_Finetuning/extended/checkpoint-1250
Checkpoint validation: PASSED

EXTENDED MODEL TRANSLATIONS

English: I never make a speech without being nervous.
Reference: Je ne fais jamais un discours sans être nerveux.
Extended model: Je ne fais jamais de discours sans être nerveuse.
------------------------------------------------------------

English: She's fashionable.
Reference: Elle est à la mode.
Extended model: Elle est à la mode.
------------------------------------------------------------

English: Remember to lock the door.
Reference: Rappelle-toi de fermer la porte à clé.
Extended model: N'oubliez pas de verrouiller la porte.
------------------------------------------------------------


In [ ]:

import pandas as pd
import sacrebleu
import torch
import time

RESULTS_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/pretrained_vs_finetuned_validation.csv"
)

three_model_df = pd.read_csv(RESULTS_URL)

expected_validation = val_pairs.drop_duplicates(
    subset="en_key"
).sample(
    n=500,
    random_state=42
).reset_index(drop=True)

assert len(three_model_df) == 500

assert (
    three_model_df["en"].tolist()
    == expected_validation["en"].tolist()
)

assert (
    three_model_df["fr"].tolist()
    == expected_validation["fr"].tolist()
)

sentences = three_model_df["en"].tolist()
references = three_model_df["fr"].tolist()

extended_model = extended_trainer.model
extended_model.eval()

model_device = next(extended_model.parameters()).device

def translate_extended(sentences, batch_size=8):
    predictions = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        ).to(model_device)

        with torch.inference_mode():
            outputs = extended_model.generate(
                **inputs,
                num_beams=1,
                do_sample=False,
                max_new_tokens=64
            )

        predictions.extend(
            tokenizer.batch_decode(
                outputs,
                skip_special_tokens=True
            )
        )

    return predictions

translate_extended(sentences[:8])

if model_device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

extended_predictions = translate_extended(sentences)

if model_device.type == "cuda":
    torch.cuda.synchronize()

extended_inference_time = time.perf_counter() - start_time

three_model_df["extended_translation"] = extended_predictions

model_columns = {
    "Original Pretrained": "transformer_greedy",
    "Fine-tuned 100 Steps": "finetuned_translation",
    "Extended Fine-tuned": "extended_translation"
}

results = []

for name, column in model_columns.items():
    predictions = three_model_df[column].tolist()

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score

    results.append({
        "Model": name,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2)
    })

three_model_results_df = pd.DataFrame(results)

print("THREE-MODEL COMPARISON")
print("Validation sentences:", len(three_model_df))
print("Decoding:", "Greedy")

display(three_model_results_df)

pilot_predictions = three_model_df["finetuned_translation"]
original_predictions = three_model_df["transformer_greedy"]

print("\nTRANSLATION DIFFERENCES")

print(
    "Extended vs Pilot:",
    (three_model_df["extended_translation"] != pilot_predictions).sum()
)

print(
    "Extended vs Original:",
    (three_model_df["extended_translation"] != original_predictions).sum()
)

print("\nEXTENDED MODEL INFERENCE")

print("Device:", model_device)
print("Time:", round(extended_inference_time, 2), "seconds")

print(
    "Sentences per second:",
    round(len(sentences) / extended_inference_time, 2)
)

changed_examples = three_model_df[
    three_model_df["extended_translation"]
    != three_model_df["finetuned_translation"]
]

display(
    changed_examples[
        [
            "en",
            "fr",
            "finetuned_translation",
            "extended_translation"
        ]
    ].head(10)
)

three_model_df.to_csv(
    "three_model_validation_predictions.csv",
    index=False
)

three_model_results_df.to_csv(
    "three_model_validation_metrics.csv",
    index=False
)

print("\nThree-model comparison files saved locally.")


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

THREE-MODEL COMPARISON
Validation sentences: 500
Decoding: Greedy


,Model,BLEU,chrF
0,Original Pretrained,55.11,70.40
1,Fine-tuned 100 Steps,55.84,71.62
2,Extended Fine-tuned,56.56,72.71



TRANSLATION DIFFERENCES
Extended vs Pilot: 145
Extended vs Original: 202

EXTENDED MODEL INFERENCE
Device: cuda:0
Time: 23.83 seconds
Sentences per second: 20.98


,en,fr,finetuned_translation,extended_translation
0,Look up the words in your dictionary.,Cherchez les mots dans votre dictionnaire.,Regardez les mots de votre dictionnaire.,Regarde les mots de ton dictionnaire.
1,I arrived at the station ten minutes late.,Je suis arrivé à la gare dix minutes en retard.,Je suis arrivé à la gare dix minutes de retard.,Je suis arrivé à la gare dix minutes en retard.
6,You have to go.,Il faut que tu t'en ailles.,Il faut que tu partes.,Il faut que tu y ailles.
8,I made sure of that.,Je m'en suis assuré.,J'en ai assuré le bien-être.,J'en ai pris soin.
14,Close the door after you when you leave the room.,Fermez la porte derrière vous quand vous quitt...,Fermez la porte après vous quand vous quittez ...,Fermez la porte après vous lorsque vous quitte...
17,I never got caught.,Je ne me suis jamais fait prendre.,Je ne me suis jamais fait attraper.,Je ne me suis jamais fait prendre.
19,I'm going to fly to Australia.,Je vais m'envoler pour l'Australie.,Je vais aller en Australie.,Je vais voler en Australie.
26,No one knows anything.,Personne ne sait quoi que ce soit.,Personne ne sait rien.,Personne ne sait quoi que ce soit.
28,I'm fed up with her.,J'en ai marre d'elle.,J'en ai marre d'elle.,J'en ai assez d'elle.
30,I think that's enough for now.,Je pense que c'est assez pour le moment.,Je pense que ça suffit pour l'instant.,Je pense que c'est suffisant pour l'instant.



Three-model comparison files saved locally.


In [ ]:

from google.colab import files

three_model_df.to_csv(
    "three_model_validation_predictions.csv",
    index=False
)

three_model_results_df.to_csv(
    "three_model_validation_metrics.csv",
    index=False
)

print("THREE-MODEL RESULTS SAVED")

print("Validation sentences:", len(three_model_df))
print("Models compared:", len(three_model_results_df))

print("\nFINAL VALIDATION METRICS")
display(three_model_results_df)

files.download("three_model_validation_predictions.csv")
files.download("three_model_validation_metrics.csv")


THREE-MODEL RESULTS SAVED
Validation sentences: 500
Models compared: 3

FINAL VALIDATION METRICS


,Model,BLEU,chrF
0,Original Pretrained,55.11,70.40
1,Fine-tuned 100 Steps,55.84,71.62
2,Extended Fine-tuned,56.56,72.71


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import numpy as np
import pandas as pd
import sacrebleu
import time

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/three_model_validation_predictions.csv"
)

stat_df = pd.read_csv(DATA_URL)

stat_df = stat_df.merge(
    val_pairs[["en", "fr", "group_id"]],
    on=["en", "fr"],
    how="left",
    validate="many_to_one"
)

assert len(stat_df) == 500
assert stat_df["group_id"].notna().all()

references = stat_df["fr"].tolist()

predictions = {
    "Original": stat_df["transformer_greedy"].tolist(),
    "Pilot": stat_df["finetuned_translation"].tolist(),
    "Extended": stat_df["extended_translation"].tolist()
}

bleu_metric = sacrebleu.metrics.BLEU()
chrf_metric = sacrebleu.metrics.CHRF()

group_rows = list(
    stat_df.groupby("group_id", sort=False).indices.values()
)

rng = np.random.default_rng(42)

N_BOOTSTRAP = 1000

def calculate_differences(indices):
    refs = [references[i] for i in indices]

    scores = {}

    for name, translations in predictions.items():
        selected = [translations[i] for i in indices]

        scores[name] = {
            "BLEU": bleu_metric.corpus_score(
                selected, [refs]
            ).score,
            "chrF": chrf_metric.corpus_score(
                selected, [refs]
            ).score
        }

    return np.array([
        [
            scores["Extended"]["BLEU"] - scores["Original"]["BLEU"],
            scores["Extended"]["chrF"] - scores["Original"]["chrF"]
        ],
        [
            scores["Extended"]["BLEU"] - scores["Pilot"]["BLEU"],
            scores["Extended"]["chrF"] - scores["Pilot"]["chrF"]
        ]
    ])

observed = calculate_differences(
    np.arange(len(stat_df))
)

bootstrap_samples = np.zeros(
    (N_BOOTSTRAP, 2, 2)
)

start_time = time.perf_counter()

for iteration in range(N_BOOTSTRAP):
    sampled_groups = rng.integers(
        0,
        len(group_rows),
        size=len(group_rows)
    )

    indices = np.concatenate([
        group_rows[i] for i in sampled_groups
    ])

    bootstrap_samples[iteration] = calculate_differences(
        indices
    )

ci_lower = np.percentile(
    bootstrap_samples, 2.5, axis=0
)

ci_upper = np.percentile(
    bootstrap_samples, 97.5, axis=0
)

rows = []

comparison_names = [
    "Extended vs Original",
    "Extended vs Pilot"
]

metric_names = ["BLEU", "chrF"]

for i, comparison in enumerate(comparison_names):
    for j, metric in enumerate(metric_names):
        rows.append({
            "Comparison": comparison,
            "Metric": metric,
            "Improvement": round(observed[i, j], 3),
            "95% CI Lower": round(ci_lower[i, j], 3),
            "95% CI Upper": round(ci_upper[i, j], 3),
            "CI Excludes Zero": bool(
                ci_lower[i, j] > 0 or ci_upper[i, j] < 0
            )
        })

extended_bootstrap_df = pd.DataFrame(rows)

print("EXTENDED MODEL STATISTICAL ANALYSIS")
print("Validation sentences:", len(stat_df))
print("Independent groups:", len(group_rows))
print("Bootstrap repetitions:", N_BOOTSTRAP)

display(extended_bootstrap_df)

print(
    "Analysis time:",
    round(time.perf_counter() - start_time, 2),
    "seconds"
)

extended_bootstrap_df.to_csv(
    "extended_model_bootstrap_results.csv",
    index=False
)


EXTENDED MODEL STATISTICAL ANALYSIS
Validation sentences: 500
Independent groups: 496
Bootstrap repetitions: 1000


,Comparison,Metric,Improvement,95% CI Lower,95% CI Upper,CI Excludes Zero
0,Extended vs Original,BLEU,1.450,-0.413,3.188,False
1,Extended vs Original,chrF,2.312,1.054,3.605,True
2,Extended vs Pilot,BLEU,0.725,-0.819,2.400,False
3,Extended vs Pilot,chrF,1.094,0.138,2.121,True


Analysis time: 302.41 seconds


In [ ]:

from google.colab import files

files.download("extended_model_bootstrap_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import pandas as pd
import sacrebleu
from google.colab import files

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/three_model_validation_predictions.csv"
)

review_source = pd.read_csv(DATA_URL)

assert len(review_source) == 500

review_source["original_chrf"] = [
    sacrebleu.sentence_chrf(pred, [ref]).score
    for pred, ref in zip(
        review_source["transformer_greedy"],
        review_source["fr"]
    )
]

review_source["extended_chrf"] = [
    sacrebleu.sentence_chrf(pred, [ref]).score
    for pred, ref in zip(
        review_source["extended_translation"],
        review_source["fr"]
    )
]

review_source["chrf_difference"] = (
    review_source["extended_chrf"]
    - review_source["original_chrf"]
)

changed = review_source[
    review_source["transformer_greedy"]
    != review_source["extended_translation"]
]

unchanged = review_source[
    review_source["transformer_greedy"]
    == review_source["extended_translation"]
]

improved_ids = changed.nlargest(
    10, "chrf_difference"
).index.tolist()

regressed_pool = changed.drop(index=improved_ids)

regressed_ids = regressed_pool.nsmallest(
    10, "chrf_difference"
).index.tolist()

remaining_changed = changed.drop(
    index=improved_ids + regressed_ids
)

random_changed_ids = remaining_changed.sample(
    n=10,
    random_state=42
).index.tolist()

unchanged_ids = unchanged.sample(
    n=10,
    random_state=42
).index.tolist()

selected_ids = (
    improved_ids
    + regressed_ids
    + random_changed_ids
    + unchanged_ids
)

selection_reasons = (
    ["Largest chrF gain"] * 10
    + ["Largest chrF decrease"] * 10
    + ["Random changed"] * 10
    + ["Random unchanged"] * 10
)

semantic_review_df = review_source.loc[
    selected_ids,
    [
        "en",
        "fr",
        "transformer_greedy",
        "finetuned_translation",
        "extended_translation",
        "original_chrf",
        "extended_chrf",
        "chrf_difference"
    ]
].copy()

semantic_review_df["selection_reason"] = selection_reasons
semantic_review_df["original_meaning_preserved"] = ""
semantic_review_df["pilot_meaning_preserved"] = ""
semantic_review_df["extended_meaning_preserved"] = ""
semantic_review_df["change_assessment"] = ""
semantic_review_df["error_categories"] = ""
semantic_review_df["reference_issue"] = ""
semantic_review_df["review_notes"] = ""

semantic_review_df = semantic_review_df.reset_index(drop=True)

semantic_review_df.to_csv(
    "three_model_semantic_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("THREE-MODEL SEMANTIC REVIEW")
print("Total examples:", len(semantic_review_df))

print("\nSelection categories:")
print(semantic_review_df["selection_reason"].value_counts())

print("\nFIRST FIVE EXAMPLES")

display(
    semantic_review_df[
        [
            "en",
            "fr",
            "transformer_greedy",
            "finetuned_translation",
            "extended_translation",
            "chrf_difference"
        ]
    ].head(5)
)

files.download("three_model_semantic_review.csv")


THREE-MODEL SEMANTIC REVIEW
Total examples: 40

Selection categories:
selection_reason
Largest chrF gain        10
Largest chrF decrease    10
Random changed           10
Random unchanged         10
Name: count, dtype: int64

FIRST FIVE EXAMPLES


,en,fr,transformer_greedy,finetuned_translation,extended_translation,chrf_difference
0,All of us like her.,Nous l'aimons tous.,Nous tous comme elle.,Nous tous comme elle.,Nous l'aimons tous.,70.904126
1,Would you care to bet?,Voudriez-vous parier ?,Tu veux bien parier ?,Voudriez-vous parier ?,Voudriez-vous parier ?,68.932665
2,This is so easy.,C'est tellement facile.,C'est si facile.,C'est si facile.,C'est tellement facile.,60.667569
3,I promise I'll try.,Je promets que j'essaierai.,Je promets d'essayer.,Je promets que j'essaierai.,Je promets que j'essaierai.,54.259490
4,We climbed right up to the top.,Nous sommes montés jusqu'au sommet.,On a grimpé jusqu'au sommet.,Nous avons grimpé jusqu'au sommet.,Nous sommes montés jusqu'au sommet.,52.398304


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import gc
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "MeouchyCharbel/Neural-Machine-Translation/"
    "feature/evaluation-improvements/"
    "results/three_model_validation_predictions.csv"
)

benchmark_df = pd.read_csv(DATA_URL)

assert len(benchmark_df) == 500
assert torch.cuda.is_available()

sentences = benchmark_df["en"].tolist()

BATCH_SIZE = 8
NUM_TRIALS = 3

model_sources = {
    "Original Pretrained": "Helsinki-NLP/opus-mt-en-fr",
    "Fine-tuned 100 Steps": str(
        Path("/content/drive/MyDrive/NMT_Finetuning/pilot/checkpoint-100")
    ),
    "Extended Fine-tuned": str(
        Path("/content/drive/MyDrive/NMT_Finetuning/extended/checkpoint-1250")
    )
}

expected_columns = {
    "Original Pretrained": "transformer_greedy",
    "Fine-tuned 100 Steps": "finetuned_translation",
    "Extended Fine-tuned": "extended_translation"
}

for name, source in model_sources.items():
    if name != "Original Pretrained":
        assert Path(source).is_dir(), f"Checkpoint missing: {source}"

def benchmark_translate(current_model, texts):
    predictions = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        ).to("cuda")

        with torch.inference_mode():
            outputs = current_model.generate(
                **inputs,
                num_beams=1,
                do_sample=False,
                max_new_tokens=64
            )

        predictions.extend(
            tokenizer.batch_decode(
                outputs,
                skip_special_tokens=True
            )
        )

    return predictions

benchmark_rows = []

for name, source in model_sources.items():
    print("\nBENCHMARKING:", name)

    current_model = AutoModelForSeq2SeqLM.from_pretrained(
        source
    ).to("cuda")

    current_model.eval()

    for _ in range(2):
        benchmark_translate(
            current_model,
            sentences[:BATCH_SIZE]
        )

    trial_times = []
    first_predictions = None

    for trial in range(NUM_TRIALS):
        torch.cuda.synchronize()

        start_time = time.perf_counter()

        predictions = benchmark_translate(
            current_model,
            sentences
        )

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time

        trial_times.append(elapsed)

        if first_predictions is None:
            first_predictions = predictions

        print(
            f"Trial {trial + 1}: {elapsed:.2f} seconds"
        )

    expected = benchmark_df[
        expected_columns[name]
    ].tolist()

    mismatches = sum(
        prediction != reference
        for prediction, reference in zip(
            first_predictions,
            expected
        )
    )

    median_time = float(np.median(trial_times))

    benchmark_rows.append({
        "Model": name,
        "Median Time (s)": round(median_time, 2),
        "Average per Sentence (ms)": round(
            median_time / len(sentences) * 1000,
            2
        ),
        "Sentences per Second": round(
            len(sentences) / median_time,
            2
        ),
        "Prediction Mismatches": mismatches
    })

    del current_model
    gc.collect()
    torch.cuda.empty_cache()

gpu_benchmark_df = pd.DataFrame(
    benchmark_rows
)

print("\nCONTROLLED GPU BENCHMARK")
print("GPU:", torch.cuda.get_device_name(0))
print("Sentences:", len(sentences))
print("Batch size:", BATCH_SIZE)
print("Trials per model:", NUM_TRIALS)

display(gpu_benchmark_df)

gpu_benchmark_df.to_csv(
    "gpu_benchmark_results.csv",
    index=False
)

print("\nBenchmark results saved.")



BENCHMARKING: Original Pretrained


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Trial 1: 10.77 seconds


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Trial 2: 8.00 seconds


... (14 outputs left out)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Trial 3: 9.08 seconds

CONTROLLED GPU BENCHMARK
GPU: Tesla T4
Sentences: 500
Batch size: 8
Trials per model: 3


,Model,Median Time (s),Average per Sentence (ms),Sentences per Second,Prediction Mismatches
0,Original Pretrained,8.72,17.43,57.37,0
1,Fine-tuned 100 Steps,8.78,17.56,56.94,1
2,Extended Fine-tuned,9.08,18.16,55.06,2



Benchmark results saved.


In [ ]:

from google.colab import files

assert len(gpu_benchmark_df) == 3

gpu_benchmark_df.to_csv(
    "gpu_benchmark_results.csv",
    index=False
)

print("GPU BENCHMARK SAVED")
display(gpu_benchmark_df)

files.download("gpu_benchmark_results.csv")


GPU BENCHMARK SAVED


,Model,Median Time (s),Average per Sentence (ms),Sentences per Second,Prediction Mismatches
0,Original Pretrained,8.72,17.43,57.37,0
1,Fine-tuned 100 Steps,8.78,17.56,56.94,1
2,Extended Fine-tuned,9.08,18.16,55.06,2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import gc
import time
import torch
import pandas as pd
import sacrebleu

from pathlib import Path
from transformers import AutoModelForSeq2SeqLM

FINAL_TEST_SIZE = 2000
BATCH_SIZE = 8

final_test_df = test_pairs.drop_duplicates(
    subset="en_key"
).sample(
    n=FINAL_TEST_SIZE,
    random_state=2026
).reset_index(drop=True)

assert len(final_test_df) == FINAL_TEST_SIZE

assert set(final_test_df["group_id"]).isdisjoint(
    set(train_pairs["group_id"])
)

assert set(final_test_df["group_id"]).isdisjoint(
    set(val_pairs["group_id"])
)

model_sources = {
    "Original Pretrained": "Helsinki-NLP/opus-mt-en-fr",
    "Extended Fine-tuned": (
        "/content/drive/MyDrive/"
        "NMT_Finetuning/extended/checkpoint-1250"
    )
}

assert Path(model_sources["Extended Fine-tuned"]).is_dir()
assert torch.cuda.is_available()

sentences = final_test_df["en"].tolist()
references = final_test_df["fr"].tolist()

def translate_test_model(current_model, texts):
    predictions = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        ).to("cuda")

        with torch.inference_mode():
            outputs = current_model.generate(
                **inputs,
                num_beams=1,
                do_sample=False,
                max_new_tokens=64
            )

        predictions.extend(
            tokenizer.batch_decode(
                outputs,
                skip_special_tokens=True
            )
        )

    return predictions

test_results = []

for name, model_path in model_sources.items():
    print("\nEVALUATING:", name)

    current_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path
    ).to("cuda")

    current_model.eval()

    translate_test_model(current_model, sentences[:BATCH_SIZE])

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    predictions = translate_test_model(
        current_model,
        sentences
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    column = (
        "original_translation"
        if name == "Original Pretrained"
        else "extended_translation"
    )

    final_test_df[column] = predictions

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score

    test_results.append({
        "Model": name,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2),
        "GPU Time (s)": round(elapsed, 2)
    })

    del current_model
    gc.collect()
    torch.cuda.empty_cache()

final_test_results_df = pd.DataFrame(test_results)

print("\nFINAL HELD-OUT TEST RESULTS")
print("Test sentences:", len(final_test_df))
print("Decoding: Greedy")
print("GPU:", torch.cuda.get_device_name(0))

display(final_test_results_df)

print("\nTRANSLATION QUALITY IMPROVEMENT")

print(
    "BLEU:",
    round(
        final_test_results_df.iloc[1]["BLEU"]
        - final_test_results_df.iloc[0]["BLEU"],
        2
    )
)

print(
    "chrF:",
    round(
        final_test_results_df.iloc[1]["chrF"]
        - final_test_results_df.iloc[0]["chrF"],
        2
    )
)

final_test_df[
    [
        "en",
        "fr",
        "original_translation",
        "extended_translation"
    ]
].to_csv(
    "final_test_predictions.csv",
    index=False
)

final_test_results_df.to_csv(
    "final_test_metrics.csv",
    index=False
)

print("\nFinal test files saved locally.")



EVALUATING: Original Pretrained


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


EVALUATING: Extended Fine-tuned


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


FINAL HELD-OUT TEST RESULTS
Test sentences: 2000
Decoding: Greedy
GPU: Tesla T4


,Model,BLEU,chrF,GPU Time (s)
0,Original Pretrained,52.22,68.03,38.05
1,Extended Fine-tuned,55.74,71.58,34.85



TRANSLATION QUALITY IMPROVEMENT
BLEU: 3.52
chrF: 3.55

Final test files saved locally.


In [ ]:

from google.colab import files

assert len(final_test_df) == 2000
assert len(final_test_results_df) == 2

files.download("final_test_predictions.csv")
files.download("final_test_metrics.csv")

print("FINAL TEST RESULTS EXPORTED SUCCESSFULLY")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FINAL TEST RESULTS EXPORTED SUCCESSFULLY
